# Loading Quantum LDPC Codes

In [4]:
import os
import pickle
from scipy.sparse import csr_matrix

# Define constants for file paths
SAVED_FILE_DIR = "/Users/aparnagupta/Downloads/notebooks/HGP/data/qLDPC_codes"
SAVED_FILE_NAME = "hgp_625.pkl"
FILE_PATH = os.path.join(SAVED_FILE_DIR, SAVED_FILE_NAME)

# Load the saved matrices
with open(FILE_PATH, "rb") as file:
    H, hgp_x, hgp_z = pickle.load(file)

EOFError: Ran out of input

# Decoding of qLDPC Codes

In [2]:
import numpy as np
import ldpc.mod2
from ldpc.bplsd_decoder import BpLsdDecoder

from lib.mod2 import quotient_basis_sp

kernel_x = ldpc.mod2.kernel(hgp_x)
logical_z = quotient_basis_sp(hgp_z, kernel_x)

bp_osd = BpLsdDecoder(
            hgp_z,
            error_rate = 0.1,
            bp_method = 'product_sum',
            max_iter = 2,
            schedule = 'serial',
            lsd_method = 'lsd_cs',
            lsd_order = 0
        )

ModuleNotFoundError: No module named 'ldpc'

In [ ]:
import numpy as np
from lib.mod2 import is_orthogonal_sp, mod2_matrix_sp

# Generate a random bit-flip error
p_err = 0.05
errors = np.random.binomial(1, p_err, size=hgp_z.shape[1])

# Calculate the corresponding syndrome and decoded error
error_syndrome = hgp_z @ errors % 2
decoded_errors = bp_osd.decode(error_syndrome)

# Combine the original and decoded errors
residual_errors = mod2_matrix_sp(errors + decoded_errors)

# Check for decoding failure
if is_orthogonal_sp(logical_z, residual_errors):
    print("Decoding Succeeded.")
else:
    print("Decoding Failed.")

# Monte-Carlo Simulation

In [ ]:
import numpy as np
import ldpc.codes
import ldpc.mod2
from ldpc import BpOsdDecoder
from ldpc.bplsd_decoder import BpLsdDecoder

from lib.mod2 import quotient_basis_sp
from lib.monte_carlo import MonteCarloBP

error_rate = 0.03

kernel_x = ldpc.mod2.kernel(hgp_x)
logical_z = quotient_basis_sp(hgp_z, kernel_x)

# Simulate the bit-flip (X) errors
parity_check_matrix = hgp_z.toarray()
logical_z_matrix = logical_z.toarray()

bp_osd = BpOsdDecoder(
            parity_check_matrix,
            error_rate = error_rate,
            bp_method = 'product_sum',
            max_iter = 7,
            schedule = 'serial',
            osd_method = 'osd_cs',
            osd_order = 2
        )

mc_sim = MonteCarloBP(parity_check_matrix, logical_z_matrix, error_rate=error_rate, Decoder=bp_osd, target_run_count=10000)
mc_sim.run()

In [ ]:
bp_lsd = BpLsdDecoder(
            hgp_z,
            error_rate = error_rate,
            bp_method = 'product_sum',
            max_iter = 2,
            schedule = 'serial',
            lsd_method = 'lsd_cs',
            lsd_order = 0
        )

mc_sim = MonteCarloBP(parity_check_matrix, logical_z_matrix, error_rate=error_rate, Decoder=bp_lsd, target_run_count=10000)
mc_sim.run()